# 09 — Model Interpretation

Use SHAP to explain individual predictions and identify which features drive demand.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib, yaml, warnings
warnings.filterwarnings('ignore')
shap.initjs()

plt.rcParams.update({'figure.facecolor':'#0f172a','axes.facecolor':'#1e293b',
    'axes.edgecolor':'#334155','axes.labelcolor':'#e2e8f0','xtick.color':'#94a3b8',
    'ytick.color':'#94a3b8','text.color':'#e2e8f0','grid.color':'#334155'})
PALETTE = ['#38bdf8','#fb7185','#34d399','#fbbf24','#a78bfa','#f97316']

with open('../configs/paths.yaml') as f:
    paths = yaml.safe_load(f)
with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

In [ ]:
# ── Load data & model ─────────────────────────────────────────────────────────
df = pd.read_csv(f'../{paths["data"]["processed"]}data_engineered.csv')
TARGET = cfg['project']['target']
X = df.drop(columns=[TARGET])
y = df[TARGET]

TEST_SIZE = cfg['project']['test_size']
split_idx = int(len(X) * (1 - TEST_SIZE))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

model = joblib.load(f'../{paths["artifacts"]["models"]}tuned_model.pkl')
model.fit(X_train, y_train)  # re-fit if needed
print('Model loaded:', type(model).__name__)

In [ ]:
# ── SHAP TreeExplainer ────────────────────────────────────────────────────────
explainer   = shap.TreeExplainer(model)
shap_values = explainer(X_test)
print('SHAP values shape:', shap_values.values.shape)

In [ ]:
# ── SHAP Summary plot (bar) ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
shap.plots.bar(shap_values, show=False, ax=ax)
ax.set_title('Mean |SHAP| Feature Importance', color='#e2e8f0')
plt.tight_layout()
plt.savefig('../outputs/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── SHAP Beeswarm plot ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
shap.plots.beeswarm(shap_values, show=False)
plt.title('SHAP Beeswarm — Feature Impact Distribution', color='#e2e8f0')
plt.tight_layout()
plt.savefig('../outputs/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── SHAP Waterfall for best/worst predictions ─────────────────────────────────
residuals = np.abs(y_test.values - model.predict(X_test))
best_idx  = np.argmin(residuals)
worst_idx = np.argmax(residuals)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, idx, label in [(axes[0], best_idx, 'Best prediction'), (axes[1], worst_idx, 'Worst prediction')]:
    plt.sca(ax)
    shap.plots.waterfall(shap_values[idx], show=False)
    ax.set_title(f'{label}  (idx={idx})', color='#e2e8f0')
plt.tight_layout()
plt.savefig('../outputs/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── SHAP Dependence plots for top features ────────────────────────────────────
shap_df  = pd.DataFrame(shap_values.values, columns=X_test.columns)
mean_abs = shap_df.abs().mean().sort_values(ascending=False)
top_feats = mean_abs.head(4).index.tolist()
print('Top features by SHAP:', top_feats)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('SHAP Dependence Plots — Top 4 Features')
for ax, feat in zip(axes, top_feats):
    ax.scatter(X_test[feat], shap_values.values[:, X_test.columns.tolist().index(feat)],
               alpha=0.5, color=PALETTE[0], s=20)
    ax.axhline(0, color='white', linestyle='--', linewidth=1.5)
    ax.set(title=feat, xlabel=feat, ylabel='SHAP value')
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('../outputs/shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Model's built-in feature importance (tree models) ─────────────────────────
if hasattr(model, 'feature_importances_'):
    fi = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(fi.index, fi.values, color=PALETTE[0], edgecolor='#0f172a')
    ax.set(title='Built-in Feature Importance (impurity / gain)',
           xlabel='Importance', ylabel='')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../outputs/feature_importance_builtin.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No built-in feature_importances_ for this model type.')

## Interpretation Summary

| Feature | Expected SHAP Direction | Business Interpretation |
|---|---|---|
| `temp` / `atemp` | ↑ | Warmer = more riders |
| `yr` | ↑ | Year-on-year brand growth |
| `season` | ↑ summer/fall | Strong seasonal peak |
| `weathersit` | ↓ | Bad weather suppresses demand |
| `hum` | ↓ | High humidity uncomfortable |
| `is_bad_weather` | ↓ | Confirms weather flag utility |

**Next:** `10_Inference.ipynb`
